# TinyML Face Expression Detection

This CoLab will walk you through creating, training, and deploying a model to classify happy and sad expressions on a microcontroller.  Thanks to former student Joshua Cheruvelil for the colab and code port.  Some modifications and fixes for versioning have been made.

https://github.com/jcheruvelil/FaceExpressionTinyML

This colab should implement a "happy vs. sad" facial expression detector on the Arduino board.



In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from tqdm import tqdm

import os, glob, json, shutil
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split

# ── Paths (edit these if your Drive folder is different) ──
CSV_DIR   = '/content/drive/MyDrive/CPE470/candy/Annotation CSV Files'
IMAGE_DIR = '/content/drive/MyDrive/CPE470/candy/Images'
OUTPUT_DIR = '/content/selected_data'

# ── Class names exactly as they appear in the CSV ──
selected_classes = [
    'Butterfingers', 'Crunch', 'Baby_Ruth', '100_Grand',
    'Snickers', 'Twix', '3_Musketeers', 'Milky_Way', 'Midnight_Milky_Way'
]

# ── Create output folders ──
for split in ['train', 'test']:
    for cls in selected_classes:
        os.makedirs(os.path.join(OUTPUT_DIR, split, cls), exist_ok=True)

# ── Read all CSVs into one dataframe ──
all_csvs = glob.glob(os.path.join(CSV_DIR, '*.csv'))
df = pd.concat([pd.read_csv(f) for f in all_csvs], ignore_index=True)
print(f"Total annotations loaded: {len(df)}")

# ── Parse bounding boxes and crop ──
crop_paths = {cls: [] for cls in selected_classes}
skipped = 0

for _, row in tqdm(df.iterrows(), total=len(df)):
    candy_type = json.loads(row['region_attributes']).get('candy_type')
    if candy_type not in selected_classes:
        skipped += 1
        continue

    shape = json.loads(row['region_shape_attributes'])
    x, y, w, h = shape['x'], shape['y'], shape['width'], shape['height']

    # Skip tiny/bad boxes (like the 26x21 outlier in the data)
    if w < 50 or h < 50:
        skipped += 1
        continue

    img_path = os.path.join(IMAGE_DIR, row['filename'])
    if not os.path.exists(img_path):
        skipped += 1
        continue

    img = Image.open(img_path).convert('RGB')
    crop = img.crop((x, y, x + w, y + h)).resize((96, 96))

    save_name = f"{row['filename'].replace('.JPG','').replace('.jpg','')}_{shape['x']}_{shape['y']}.jpg"
    save_path = os.path.join('/content/crops', candy_type, save_name)
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    crop.save(save_path)
    crop_paths[candy_type].append(save_path)

print(f"Skipped {skipped} annotations (bad boxes or missing images)")

# ── Split into train/test (80/20) and copy ──
for cls in selected_classes:
    files = crop_paths[cls]
    if len(files) == 0:
        print(f"WARNING: No crops found for {cls}")
        continue
    train_files, test_files = train_test_split(files, test_size=0.2, random_state=42)
    for f in train_files:
        shutil.copy(f, os.path.join(OUTPUT_DIR, 'train', cls, os.path.basename(f)))
    for f in test_files:
        shutil.copy(f, os.path.join(OUTPUT_DIR, 'test', cls, os.path.basename(f)))
    print(f"{cls}: {len(train_files)} train, {len(test_files)} test")

print("\nDone! Crops saved to /content/selected_data/")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total annotations loaded: 2113


 52%|█████▏    | 1089/2113 [04:17<03:53,  4.38it/s]

In [ ]:
import shutil
shutil.copytree('/content/selected_data', '/content/drive/MyDrive/CPE470/candy/selected_data', dirs_exist_ok=True)
print("Backed up to Drive!")

###Hardware Needed:

*   Arduino Nano 33 BLE Sense
*   0V7675 Camera

(Both contained in the TinyML kit)



# 1. Prepare Dataset


We will be using the Facial Expression Recognition dataset (FER) from Kaggle. This dataset contains images for 7 expression categories, but we will only be focusing on 2 expressions - happy and sad.
1.   Dowload the dataset from https://www.kaggle.com/datasets/msambare/fer2013
2.   Upload the downloaded zipfile to this CoLab using the Files side menu to the left
3.   Run the following code cell to unzip the file and extract the happy and sad classes from the dataset. Be sure to define your uploaded file path.

note, I had to restart the session to resolve some dependencies. (JO: 5.7.2026)


In [ ]:
!pip install -U tf_keras
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

# Section 0 already created these — just define the paths here
selected_train_data_dir = '/content/selected_data/train'
selected_test_data_dir  = '/content/selected_data/test'

selected_classes = [
    'Butterfingers', 'Crunch', 'Baby_Ruth', '100_Grand',
    'Snickers', 'Twix', '3_Musketeers', 'Milky_Way', 'Midnight_Milky_Way'
]

# 2. Transfer Learning Approach

Our first attempt at a model will be to use a pretrained model and modify the output layer to match our data. This is a **transfer learning** approach.

Let's try the [MobileNetV3 model](https://keras.io/api/applications/mobilenet/). Since this model is trained on 224x224 color images, we need to make sure to make sure the inputs to the model match this.

Preprocessing - Organize the data into train and test generators using 224x224 RGB images and the mobilenetv3 preprocessing function

Note, it looks like ths model is going to be toolarge

###Let's Assess:
This model looks to be pretty accurate with an accuracy of 80-90% after 20 epochs. However....

There are 2 main problems with using this model:
1. Size - If we look above to where we printed the model summary, we can see the model size is around 1.7 megabytes. Our Nano 33 BLE has a flash memory size of 1 megabyte. Even though the conversion to TFLite will reduce this size, we probably will need a much smaller model for deploying on our board.

2. TFLite Supported Layers - TensorFlow Lite for Microcontrollers only supports a handful of model layers in the op_resolver which is needed while deploying. The supported ops can be found [here](https://github.com/tensorflow/tflite-micro/blob/main/tensorflow/lite/micro/micro_mutable_op_resolver.h). If we look into the architecture of the MobileNet model, there are some layers that are not supported by TFLite which would cause us to run into some problems during deployment.

Let's pivot to a different approach - building a simpler model from scratch.

# 3. Simpler Model From Scratch


Let's build our own model with the input being 96x96 grayscale images and using 4 convolutional blocks of increasing sizes. We then include a flatten layer and two Dense layers with an output layer of size 2, corresponding to the 2 features we have (happy, sad).

JO: TODO - add depthwise convolution  (DONE)
https://www.tensorflow.org/api_docs/python/tf/keras/layers/DepthwiseConv2D
Add comparison between this and regular CNN



In [ ]:
from tensorflow.keras import layers, models

### DEFINE THE MODEL

model = models.Sequential()

# Input Layer
model.add(layers.Input(shape=(64, 64, 3)))

# Convolutional Block 1
model.add(layers.Conv2D(16, (3, 3), activation='relu', padding='same'))
model.add(layers.MaxPooling2D((2, 2)))  # Reduces size by half

# Convolutional Block 2
model.add(layers.Conv2D(32, (3, 3), activation='relu', padding='same'))
model.add(layers.MaxPooling2D((2, 2)))  # Further reduces size by half

# Convolutional Block 3
model.add(layers.Conv2D(64, (3, 3), activation='relu', padding='same'))
model.add(layers.MaxPooling2D((2, 2)))  # Further reduces size by half

# Convolutional Block 4 (Depthwise Separable with channel reduction)
model.add(layers.DepthwiseConv2D(kernel_size=(3, 3), activation='relu', padding='same')) # depth_multiplier defaults to 1
model.add(layers.Conv2D(filters=8, kernel_size=(1, 1), activation='relu', padding='same')) # Pointwise convolution to set final channels to 8 (1/4 of 32)
model.add(layers.MaxPooling2D((2, 2)))  # Further reduces size by half

# Flatten Layer
model.add(layers.Flatten())

# Fully connected layer

model.add(layers.Dense(32, activation='relu'))

# Output Layer
model.add(layers.Dense(9, activation='softmax'))


### COMPILE THE MODEL

model.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['acc'])

model.summary()


Notice, our model is a lot smaller - significantly smaller than the 1MB limit.

Preprocessing - Organize the data into train and test generators - use data augmentation for the training set.

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

image_size = (96, 96)
batch_size = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range = 10,
    width_shift_range = 10,
    height_shift_range = 10,
    shear_range = 0.2,
    zoom_range = 0.2,
    horizontal_flip = True,
    fill_mode = 'nearest'
    )
test_datagen = ImageDataGenerator(rescale=1./255)

# Load and preprocess training images
train_generator = train_datagen.flow_from_directory(
    selected_train_data_dir,
    target_size=image_size,
    batch_size=batch_size,
    class_mode='categorical',
    color_mode='rgb'
)

# Load and preprocess test images
test_generator = test_datagen.flow_from_directory(
    selected_test_data_dir,
    target_size=image_size,
    batch_size=batch_size,
    class_mode='categorical',
    color_mode='rgb'
)

This time while training the model we will need more epochs since we built the model from scratch.

In [ ]:
# Train the model with 150 epochs
history = model.fit(train_generator,
                    steps_per_epoch=len(train_generator),
                    epochs=100,
                    #epochs=5,  # for debug
                    validation_data=test_generator,
                    validation_steps=len(test_generator),
                    )

Epoch 1/100
53/53 [==============================] - 8s 99ms/step - loss: 2.1718 - acc: 0.1234 - val_loss: 2.0094 - val_acc: 0.1171
Epoch 2/100
53/53 [==============================] - 5s 90ms/step - loss: 1.7357 - acc: 0.2504 - val_loss: 1.4302 - val_acc: 0.3513
Epoch 3/100
53/53 [==============================] - 5s 89ms/step - loss: 1.2304 - acc: 0.5217 - val_loss: 1.1127 - val_acc: 0.5738
Epoch 4/100
53/53 [==============================] - 5s 90ms/step - loss: 0.9009 - acc: 0.6772 - val_loss: 0.7229 - val_acc: 0.8009
Epoch 5/100
53/53 [==============================] - 5s 91ms/step - loss: 0.5532 - acc: 0.8184 - val_loss: 0.5638 - val_acc: 0.8618
Epoch 6/100
53/53 [==============================] - 5s 90ms/step - loss: 0.3560 - acc: 0.9056 - val_loss: 0.3943 - val_acc: 0.9321
Epoch 7/100
53/53 [==============================] - 5s 90ms/step - loss: 0.2829 - acc: 0.9234 - val_loss: 0.3702 - val_acc: 0.9391
Epoch 8/100
53/53 [==============================] - 5s 89ms/step - loss: 0.

This model is not as accurate as the transfer learning approach but is a lot more appropriate for TFLite and deploying to a microcontroller. Let's move on to the TFLite conversion.

#4. Convert to TensorFlow Lite Model

The next step of the process is to convert the trained model to a TFLite model to optimize for size. During this process we will also quantize the model to use integer inputs and outputs to further optimize the model.

In [ ]:
import numpy as np


# Save Trained Model  - this is crashing
CANDY_SAVED_MODEL = "candy_saved_model"
tf.saved_model.save(model, CANDY_SAVED_MODEL)


# Convert Model using TFLiteConverter

#FER_SAVED_MODEL = '/content/fer_saved_model'
import pathlib
converter = tf.lite.TFLiteConverter.from_saved_model(CANDY_SAVED_MODEL)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
def representative_dataset_gen():
    for _ in range(100):
        # Get sample input data as a numpy array
        yield [np.random.rand(1, 96, 96, 3).astype(np.float32)]

converter.representative_dataset = representative_dataset_gen

#following line is crashing the runtime
tflite_model = converter.convert()
tflite_models_dir = pathlib.Path("/content/")

tflite_model_file = tflite_models_dir / 'candy_model.tflite'
tflite_model_file.write_bytes(tflite_model)
# This will report back the file size in bytes

This conversion reduced our model size even more. Next let's test out our converted model on a sample of 100 images and see how it performs.

In [ ]:
from tqdm import tqdm
# Load TFLite model and allocate tensors.
interpreter = tf.lite.Interpreter(model_path='/content/candy_model.tflite')
interpreter.allocate_tensors()


test_data_dir = '/content/selected_data/test'
test_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    test_data_dir,
    image_size=(96, 96),  # Adjust size as needed
    batch_size=1,          # Adjust batch size as needed
    color_mode='rgb',
    label_mode='int'        # Labels will be integers
)

# Check input and output tensor details
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

input_index = input_details[0]['index']
output_index = output_details[0]['index']

print('Input Details:', input_details)
print('Output Details:', output_details)

# Ensure the input size matches the model's input size
input_shape = input_details[0]['shape']  # Expected input shape
print('Expected Input Shape:', input_shape)

# Prepare lists to store test labels and predictions
test_labels = []
test_imgs = []
predictions = []

def preprocess_image(image):
    # Normalize and cast to int8 as required by TFLite
    normalized_image = (image / 255.0) * 2.0 - 1.0
    int8_image = np.clip(normalized_image * 127, -128, 127).astype(np.int8)
    return int8_image

test_dataset = test_dataset.take(100)

for img_batch, label_batch in tqdm(test_dataset):
    for img, label in zip(img_batch, label_batch):
        preprocessed_img = preprocess_image(img)
        interpreter.set_tensor(input_index, [preprocessed_img])
        interpreter.invoke()
        predictions.append(interpreter.get_tensor(output_index))

        test_labels.append(label.numpy())
        test_imgs.append(img.numpy())

print(len(predictions))

# Evaluate accuracy
score = 0
for item in range(len(predictions)):
    prediction = np.argmax(predictions[item])
    label = test_labels[item]
    if prediction == label:
        score += 1

print(f"Out of 100 predictions, {score} were correct")


#5. Deploying to your Board

The final step is to deploy to our Arduino Nano 33 BLE board. Let's first convert our model to TFLite Micro.


In [ ]:
MODEL_TFLITE = '/content/candy_model.tflite'
MODEL_TFLITE_MICRO = 'candy_model.cc'
!xxd -i {MODEL_TFLITE} > {MODEL_TFLITE_MICRO}
REPLACE_TEXT = MODEL_TFLITE.replace('/', '_').replace('.', '_')
!sed -i 's/'{REPLACE_TEXT}'/g_model/g' {MODEL_TFLITE_MICRO}


To Deploy:

1. Make sure you have the Arduino IDE installed and then navigate to https://github.com/jcheruvelil/FaceExpressionTinyML/tree/main/expression_detection and download the Arduino sketch.

2. Download fer_model.cc generated from the above code cell.

3. In expression_detect_model_data.cpp copy and paste just the byte array from fer_model.cc into the byte array field, and set g_expression_detect_model_data_len to the number of bytes listed at the bottom of the fer_model.cc file.

4. Make sure your 0V7675 Camera is hooked up to your Nano 33 BLE board and connect the board to your laptop via USB.

5. Compile and Upload the Arduino sketch to your board. Compilation will take a long time on your first time compiling.

6. Once uploaded, open up the Serial monitor to see the happy and sad scores for each image capture. Make sure the camera is far enough away from your face while capturing images (extend your arm straight while holding the board/camera). A green led corresponds to a happy expression while a red led corresponds to a sad expression. A blue led lights up between every image capture.

Observe how well or how poorly your model performs. Try different lightings and environments and feel free to experiment by changing things in any of the previous steps to obtain different results.

Have fun!